# Task 5

## Include

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve_triangular
import scipy.sparse as sp
import scipy.linalg as la

import subprocess                   # Library to call cpp executables
import matplotlib.pyplot as plt     # Library for spy
import time

## Method for all previous tasks

Define a method that calls the previous tasks.

In [ ]:
def esecuzioneTasks12(N=32):
    # Definiamo l'ordine di files da eseguire
    executables = [
        ["../Task1/main", f"{N}"],
        ["../Task2/main"]
    ]

    for i in range(len(executables)):
        result = subprocess.run(
            executables[i],       # Eseguiamo il file executables[i] con gli argomenti in inputs[i]
        )

def esecuzioneTask3(N=32, usaOrdering=0):
    result = subprocess.run(
        ["../Task2/main", f"{N}", f"{usaOrdering}"]
    )

# Definiamo la funzione my_cholesky come indicato nelle specifiche
def my_cholesky(A):
    # -A è definita positiva
    neg_A = -A.toarray() if sp.issparse(A) else -A
    # lower=True restituisce la matrice triangolare inferiore L tale che -A = L @ L.T
    L = la.cholesky(neg_A, lower=True)
    # Ritorna come csc_matrix sparsa
    return sp.csc_matrix(L)

def esecuzioneTask4(verbose=False):
    # Tempo di inizio algoritmo
    start_time = time.time()

    # 1. Caricamento dati e modifica
    # Caricamento del vettore rhs.txt
    rhs = np.loadtxt("rhs.txt")
    num_incognite = len(rhs)
    if verbose:
        print(f"Dimensioni del vettore rhs: {num_incognite}")

    # Caricamento della matrice sparsa A.txt
    dati_A = np.loadtxt("A.txt")
    righe = dati_A[:, 0].astype(int)
    colonne = dati_A[:, 1].astype(int)
    valori = dati_A[:, 2]
    if verbose:
        print(f"Dimensioni della matrice A: {dati_A.shape}")

    # Costruzione della matrice sparsa A in formato COO
    A_coo = sp.coo_matrix((valori, (righe, colonne)), shape=(num_incognite, num_incognite))

    # Conversione della matrice A in formato CSC (Compressed Sparse Column)
    A_csc = A_coo.tocsc()
    if verbose:
        print(f"Matrice A convertita con successo in formato CSC ({A_csc.shape[0]}x{A_csc.shape[1]}).")

    # 2. Fattorizzazione di Cholesky: -A = L * L^T
    # Eseguiamo la fattorizzazione di Cholesky per ottenere la matrice L
    L = my_cholesky(A_csc)
    if verbose:
        print("Fattorizzazione di Cholesky -A = L * L^T eseguita con successo.")
        print(f"Dimensione del fattore L: {L.shape}")

    # Nota: Dato che fattorizziamo -A, trasformiamo l'equazione A * u = rhs in (-A) * u = -rhs
    b = -rhs

    # 3a. Risoluzione del sistema triangolare inferiore L * y = b
    y = spsolve_triangular(L, b, lower=True)

    # 3b. Risoluzione del sistema triangolare superiore L^T * u = y
    L_T = L.transpose().tocsc()
    u = spsolve_triangular(L_T, y, lower=False)

    # Tempo di fine algoritmo
    end_time = time.time()

    if verbose:
        print("Sistema risolto con successo!\n")

    # 4. Verifica della soluzione ed Errore di Residuo
    residuo = np.linalg.norm(A_csc.dot(u) - rhs)
    if verbose:
        print(f"Norma del residuo ||A*u - rhs||: {residuo:.2e}")

        print("\nPrime componenti della soluzione approssimata u:")
        for idx, val in enumerate(u[:min(9, len(u))]):
            print(f"  u[{idx}] = {val:.6f}")

    ### Risultati
    tempoPassato = end_time - start_time        # Tempo di esecuzione
    entrNonZero = np.count_nonzero(A_csc)       # Numero entrate non-zero della matrice

    return (A_csc, u, residuo, tempoPassato, entrNonZero)

# Analisi sperimentale

In [19]:
sequence_N = [32, 64, 128, 256, 512, 1024]

tempiEsecuzione = {}
entrateMatrice = {}
matriceA = {}
soluzioneU = {}

# Itero per ogni N l'intero algoritmo
for ordering in [0, 1]:
    for N in sequence_N:
        # Eseguo tasks 1 e 2
        start_t = time.time()
        esecuzioneTasks12(N=N)
        time_tasks12 = time.time() - start_t

        # Eseguo task 3
        start_t = time.time()
        esecuzioneTask3(N=N, usaOrdering=ordering)
        time_task3 = time.time() - start_t

        # Eseguo task 4
        [A, u, residuo, time_task4, entrNonZero] = esecuzioneTask4()

        # Salvo i dati
        tempiEsecuzione[(N,ordering)] = [time_tasks12, time_task3, time_task4]
        entrateMatrice[(N,ordering)] = entrNonZero
        matriceA[(N,ordering)] = A
        soluzioneU[(N,ordering)] = u

FileNotFoundError: [Errno 2] No such file or directory: '../Task1/main'

# Tempi di esecuzione

In [ ]:
# Natural ordering

labels = sequence_N
values = [tempiEsecuzione[(N,0)] for N in sequence_N]

plt.bar(labels, values)
plt.xlabel("N")
plt.ylabel("Tempo Esecuzione")
plt.title("Ordering naturale")
plt.show()

# Splitting

labels = sequence_N
values = [tempiEsecuzione[(N,1)] for N in sequence_N]

plt.bar(labels, values)
plt.xlabel("N")
plt.ylabel("Tempo Esecuzione")
plt.title("Ordering naturale")
plt.show()

KeyError: (32, 0)